# アノテーションの一致度を測る（κ 係数、Krippendorff の α）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_evaluation/agreement.ipynb)

NLP クックブック 20-09 ／ 評価指標と統計的検定

複数人が付けたラベルがどれくらい一致しているかを、偶然の一致を差し引いて評価したい。

In [1]:
!pip install -q krippendorff statsmodels

In [2]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

a = ["pos", "pos", "neg", "neu", "pos", "neg", "neg", "pos", "neu", "pos", "neg", "pos"]
b = ["pos", "neu", "neg", "neu", "pos", "neg", "pos", "pos", "neg", "pos", "neg", "pos"]
print("単純一致率:", round(float(np.mean([x == y for x, y in zip(a, b)])), 3))
print("Cohen の κ:", round(cohen_kappa_score(a, b), 3))

# 順序のあるラベル (1〜5 の評点など) は、近い食い違いを軽く扱う重み付き κ を使う
r1, r2 = [1, 2, 3, 4, 5, 3, 2, 4, 5, 1], [1, 3, 3, 5, 5, 2, 2, 4, 4, 2]
print("重みなし κ:", round(cohen_kappa_score(r1, r2), 3), " 2 次重み付き κ:", round(cohen_kappa_score(r1, r2, weights="quadratic"), 3))

単純一致率: 0.75
Cohen の κ: 0.591
重みなし κ: 0.375  2 次重み付き κ: 0.865


In [3]:
# 3 人以上: Fleiss の κ
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

ratings = np.array([[0, 0, 0], [0, 1, 0], [1, 1, 1], [2, 2, 1], [0, 0, 0], [1, 1, 2], [2, 2, 2], [0, 0, 1]])   # 行 = 事例、列 = 評定者
table, _ = aggregate_raters(ratings)
print("Fleiss の κ:", round(fleiss_kappa(table), 3))

Fleiss の κ: 0.489


In [4]:
# 欠損があってもよく、名義・順序・間隔尺度に対応する Krippendorff の α
import krippendorff

data = np.array([[1, 2, 3, 3, 2, 1, 4, 1, 2, np.nan],      # 行 = 評定者、列 = 事例 (nan = 未評定)
                 [1, 2, 3, 3, 2, 2, 4, 1, 2, 5],
                 [np.nan, 3, 3, 3, 2, 3, 4, 2, 2, 5]])
for level in ["nominal", "ordinal", "interval"]:
    print(f"α ({level}):", round(krippendorff.alpha(reliability_data=data, level_of_measurement=level), 3))

α (nominal): 0.675
α (ordinal): 0.805
α (interval): 0.862


- κ の目安 (Landis & Koch): 0.41〜0.60 中程度、0.61〜0.80 かなりの一致、0.81 以上ほぼ完全。ただし絶対的な基準ではない。
- κ はラベルの分布が偏っていると、一致率が高くても低く出る。混同行列も併せて見る。
- 一致度が低いときは、アノテータではなくガイドラインに問題があることが多い。食い違った事例を見て、基準を明確にする。